[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/05_mcp.ipynb)

# Part 5 — MCP

**MCP is the Model Context Protocol**: an open standard for how an LLM application discovers
and calls tools that live outside its own process.

In Parts 1–4 every tool was a Python function in the same process as the agent loop. That is
fine for a notebook and wrong for a real simulation stack, where the solver is a compiled
binary on a cluster, behind a scheduler, written by someone else.

MCP splits that in two:

- a **server** owns the tools and publishes their names, descriptions and JSON schemas;
- a **client** connects, asks what is available, and calls what it needs.

Write your solver's tool surface once as a server, and anything that speaks MCP can drive it.
This notebook builds one server and drives it two ways: by hand, then with a model.

In [ ]:
import sys
if 'google.colab' in sys.modules:
    %pip install -U -q mcp scikit-fem "google-genai<2.13" "google-auth==2.49.0"

import os, json
from google import genai
from google.genai import types as gtypes
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

gemini = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def pick_errlog():
    """stdio_client attaches the server's stderr to a real file descriptor.
    A notebook's sys.stderr has none, so fall back to the process's original."""
    for s in (sys.__stderr__, sys.stderr):
        try:
            if s is not None:
                s.fileno()
                return s
        except Exception:
            continue
    return open(os.devnull, "w")


ERRLOG = pick_errlog()
print("ready.")

## 1. The server

A server is a plain Python file. `@mcp.tool()` publishes a function, and the SDK derives the
tool's public contract from the signature: **type hints become the JSON schema**, the
**docstring becomes the description** the model reads.

That contract is the whole product. Two things to notice below:

- `op` is typed `Literal["add", ...]`, not `str`. That produces an `enum` in the published
  schema, and the server then **rejects** anything else before your function runs. A plain
  `str` would let bad values through to your code.
- `solve_poisson` returns numbers, not a mesh. Everything crossing the boundary is JSON, so
  the mesh stays server-side and the client gets a handle-free summary.

(Tools are one of three MCP primitives. There are also *resources* — read-only data addressed
by URI — and *prompts*. This notebook only uses tools.)

In [ ]:
%%writefile aescape_mcp_server.py
from typing import Literal

from mcp.server.mcpserver import MCPServer

import numpy as np
from skfem import (MeshTri, Basis, ElementTriP1, BilinearForm, LinearForm,
                   condense, solve)
from skfem.helpers import dot, grad

mcp = MCPServer(name="aescape-tools")


@BilinearForm
def stiffness(u, v, w):
    return dot(grad(u), grad(v))


@LinearForm
def unit_load(v, w):
    return 1.0 * v


@mcp.tool()
def calculator(op: Literal["add", "sub", "mul", "div"], a: float, b: float) -> dict:
    """Perform one arithmetic operation on two numbers."""
    ops = {"add": a + b, "sub": a - b, "mul": a * b,
           "div": a / b if b != 0 else float("nan")}
    return {"result": ops[op]}


@mcp.tool()
def solve_poisson(refine: int = 3) -> dict:
    """Solve -laplace(u) = 1 on the L-shaped domain with u = 0 on the boundary.

    refine: number of uniform refinements, 0 to 6. Higher is finer and slower.
    """
    if not 0 <= refine <= 6:
        return {"error": f"refine must be between 0 and 6; got {refine}"}
    mesh = MeshTri.init_lshaped()
    for _ in range(refine):
        mesh = mesh.refined()
    basis = Basis(mesh, ElementTriP1())
    K = stiffness.assemble(basis)
    f = unit_load.assemble(basis)
    u = solve(*condense(K, f, D=mesh.boundary_nodes()))
    return {"refine": refine,
            "dofs": int(basis.N),
            "elements": int(mesh.nelements),
            "u_max": float(np.max(u))}


if __name__ == "__main__":
    mcp.run(transport="stdio")

## 2. Driving it by hand

`stdio` transport launches the server as a subprocess and talks to it over stdin/stdout.
(`streamable-http` is the alternative when the server lives on another machine.)

`list_tools()` is the discovery step: the client learns the tool surface at runtime instead of
having it hard-coded. Watch what the published schema looks like — that is what a model will
be shown.

In [ ]:
SERVER = os.path.abspath("aescape_mcp_server.py")


def connect():
    return stdio_client(
        StdioServerParameters(command=sys.executable, args=[SERVER]), errlog=ERRLOG)


async def by_hand():
    async with connect() as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            for t in (await session.list_tools()).tools:
                print(f"{t.name}\n  {(t.description or '').splitlines()[0]}")
                print(f"  schema: {json.dumps(t.input_schema['properties'])}\n")

            r = await session.call_tool("calculator", {"op": "mul", "a": 13, "b": 47})
            print("calculator(mul, 13, 47)   ->", r.content[0].text.replace("\n", " "))

            r = await session.call_tool("solve_poisson", {"refine": 4})
            print("solve_poisson(refine=4)   ->", r.content[0].text.replace("\n", " "))

            # The enum is enforced by the server: this never reaches our function.
            r = await session.call_tool("calculator", {"op": "bogus", "a": 1, "b": 2})
            print("calculator(bogus, ...)    -> is_error =", r.is_error)
            detail = next((l.strip() for l in r.content[0].text.splitlines()
                            if "Input should be" in l), "")
            print("   ", detail)

await by_hand()

## 3. Driving it with a model

Calling tools by hand is just RPC. The point of publishing a *schema* is that a model can read
it and decide for itself.

The bridge is the interesting part: an MCP `input_schema` is already JSON Schema, which is
exactly what Gemini's `FunctionDeclaration` wants — so it passes through **unmodified**. The
server's contract becomes the model's tool list with no translation layer.

This is the same ReAct loop as Part 1. The only change is where the tools come from: discovered
over a protocol instead of written in this file.

In [ ]:
async def ask(question, max_steps=6):
    async with connect() as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # MCP schemas drop straight into the model's tool list.
            decls = [gtypes.FunctionDeclaration(name=t.name,
                                                description=t.description,
                                                parameters=t.input_schema)
                     for t in (await session.list_tools()).tools]
            cfg = gtypes.GenerateContentConfig(
                system_instruction="Use the provided tools. Never compute in your head.",
                tools=[gtypes.Tool(function_declarations=decls)])

            contents = [gtypes.Content(role="user",
                                       parts=[gtypes.Part.from_text(text=question)])]
            for _ in range(max_steps):
                resp = gemini.models.generate_content(
                    model=MODEL, contents=contents, config=cfg)
                parts = resp.candidates[0].content.parts or []
                contents.append(resp.candidates[0].content)

                calls = [p.function_call for p in parts if getattr(p, "function_call", None)]
                if not calls:
                    return "".join(getattr(p, "text", "") or "" for p in parts)

                obs = []
                for fc in calls:
                    args = dict(fc.args or {})
                    print(f"  MCP call: {fc.name}({args})")
                    r = await session.call_tool(fc.name, args)
                    text = r.content[0].text
                    try:
                        payload = json.loads(text)
                    except json.JSONDecodeError:
                        payload = {"error": text}
                    obs.append(gtypes.Part.from_function_response(
                        name=fc.name, response=payload))
                contents.append(gtypes.Content(role="user", parts=obs))
            return "(max_steps reached)"


print(await ask("What is 13 * 47 + 8?"))
print()
print(await ask("Solve the Poisson problem with 5 refinements. How many degrees of freedom?"))

## Syntax summary

**Server**

| | |
|---|---|
| `from mcp.server.mcpserver import MCPServer` | the server class |
| `mcp = MCPServer(name="...")` | create it |
| `@mcp.tool()` | publish a function; hints → schema, docstring → description |
| `Literal[...]` on a parameter | becomes an `enum`; the server enforces it |
| `mcp.run(transport="stdio")` | serve over stdin/stdout (or `"streamable-http"`) |

**Client**

| | |
|---|---|
| `StdioServerParameters(command=..., args=[...])` | how to launch the server |
| `async with stdio_client(params, errlog=...)` | open the transport |
| `async with ClientSession(read, write) as session` | open a session |
| `await session.initialize()` | handshake |
| `await session.list_tools()` | discover — `.name`, `.description`, `.input_schema` |
| `await session.call_tool(name, {...})` | invoke — `.content[0].text`, `.is_error` |

Three things that will bite you:

- **The SDK renamed things in 2.x.** `FastMCP` is now `MCPServer`, and `inputSchema` is now
  `input_schema`. Most tutorials online are still 1.x. Pin `mcp<2` for the old API.
- **Never `print()` inside a stdio tool.** stdout *is* the protocol channel; a stray print
  corrupts the stream. Log to stderr instead.
- **In a notebook, pass `errlog=` explicitly.** It defaults to `sys.stderr`, which the kernel
  replaces with an object having no file descriptor — you get `UnsupportedOperation: fileno`.

## Further reading

- [modelcontextprotocol.io](https://modelcontextprotocol.io) — specification and concepts
- [Python SDK](https://github.com/modelcontextprotocol/python-sdk) — source and examples
- [2.x migration guide](https://py.sdk.modelcontextprotocol.io/v2/migration/) — what changed from 1.x
- [Reference servers](https://github.com/modelcontextprotocol/servers) — filesystem, git, databases and more